# Day 1 · Hands-on 5 — Power and sample size by simulation
**About 30 min · work in pairs** (one person drives the keyboard, the other reads, predicts and questions — swap halfway)


You are planning a trial of a new drug that **truly shrinks tumours by 5 mm on average**. Tumour sizes vary as much as in GBSG2. How many patients do you need to detect the effect reliably? We answer by simulating the trial thousands of times.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has exactly one line marked `# 🔧 CHANGE THIS` — that is the only thing you need to edit. Lines starting with 📋 are results to enter in the shared sheet.

In [ ]:
# ▶️ Run this cell first (about 20 seconds). Nothing to change here.
%pip install -q lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines.datasets import load_gbsg2

df = load_gbsg2()
print(f"✅ Loaded GBSG2: {len(df)} patients, {df.shape[1]} variables")

## A. One planned trial
### 🧭 Concept
Patients are drawn from the real GBSG2 tumour sizes; treated patients' tumours are 5 mm smaller. The effect is **real**, but will your trial find it?

Run the cell several times. How often do you get p < 0.05?

In [ ]:
# 🔧 CHANGE THIS: patients per group
n_per_group = 30

effect_mm = 5
rng = np.random.default_rng()
tsize = df["tsize"].to_numpy()
control = rng.choice(tsize, n_per_group)
treated = np.clip(rng.choice(tsize, n_per_group) - effect_mm, 1, None)   # 5 mm smaller (minimum 1 mm)

p = stats.ttest_ind(control, treated, equal_var=False).pvalue
print(f"Estimated effect: {control.mean() - treated.mean():.1f} mm   p = {p:.3f}   ",
      "✅ detected" if p < 0.05 else "❌ missed (Type II error)")

## B. Power = how often a real effect is detected
### 🧭 Concept
Running the trial 2,000 times, **power** is the fraction of trials with p < 0.05. Use the sample size your facilitator assigned.

In [ ]:
# 🔧 CHANGE THIS: patients per group — use the value assigned to your pair
n_per_group = 30

effect_mm, alpha, n_trials = 5, 0.05, 2000
rng = np.random.default_rng()
tsize = df["tsize"].to_numpy()
control = rng.choice(tsize, size=(n_trials, n_per_group))
treated = np.clip(rng.choice(tsize, size=(n_trials, n_per_group)) - effect_mm, 1, None)
p = stats.ttest_ind(control, treated, axis=1, equal_var=False).pvalue
power = np.mean(p < alpha)

print("📋 For the shared sheet:")
print(f"   n per group = {n_per_group}   |   power = {power:.0%}")

## C. The whole power curve
### 🧭 Concept
Now simulate a range of sample sizes at once. The convention is to aim for at least **80% power**. Change the effect size: what happens to the curve if the drug only shrinks tumours by 3 mm? Or by 10 mm?

In [ ]:
# 🔧 CHANGE THIS: true effect of the drug (mm)
effect_mm = 5

alpha, n_trials = 0.05, 2000
sizes = [10, 20, 30, 50, 75, 100, 150, 200, 300, 400]
rng = np.random.default_rng()
tsize = df["tsize"].to_numpy()

powers = []
for n in sizes:
    control = rng.choice(tsize, size=(n_trials, n))
    treated = np.clip(rng.choice(tsize, size=(n_trials, n)) - effect_mm, 1, None)
    powers.append(np.mean(stats.ttest_ind(control, treated, axis=1, equal_var=False).pvalue < alpha))

plt.figure(figsize=(7, 3.5))
plt.plot(sizes, powers, "o-")
plt.axhline(0.8, color="red", linestyle="--", label="80% power")
plt.ylim(0, 1.02); plt.xlabel("Patients per group"); plt.ylabel("Power"); plt.legend()
plt.title(f"Power to detect a {effect_mm} mm effect")
plt.show()
enough = [n for n, pw in zip(sizes, powers) if pw >= 0.8]
print(f"Smallest simulated size with ≥80% power: {enough[0] if enough else 'more than ' + str(sizes[-1])} per group")

## D. Check against the textbook formula
### 🧭 Concept
For a t-test there is a formula. It needs the **standardized effect** (Cohen's d = effect / SD). Compare its answer with your simulation. They should be close but not identical: the formula assumes normal data, while the simulation uses the real, skewed tumour sizes.

In [ ]:
# 🔧 CHANGE THIS: true effect of the drug (mm)
effect_mm = 5

from statsmodels.stats.power import TTestIndPower
sd = df["tsize"].std()
d = effect_mm / sd
n_needed = TTestIndPower().solve_power(effect_size=d, power=0.8, alpha=0.05)
print(f"SD of tumour size: {sd:.1f} mm   →   Cohen's d = {d:.2f}")
print(f"Formula: {np.ceil(n_needed):.0f} patients per group for 80% power")

## E. ⭐ If you have time: the winner's curse
### 🧭 Concept
In an underpowered study, which results reach p < 0.05? Only those that by luck **overestimated** the effect. Compare small and large studies.

In [ ]:
# 🔧 CHANGE THIS: patients per group — try 15, then 150
n_per_group = 15

effect_mm, n_trials = 5, 5000
rng = np.random.default_rng()
tsize = df["tsize"].to_numpy()
control = rng.choice(tsize, size=(n_trials, n_per_group))
treated = np.clip(rng.choice(tsize, size=(n_trials, n_per_group)) - effect_mm, 1, None)
estimates = control.mean(axis=1) - treated.mean(axis=1)
significant = stats.ttest_ind(control, treated, axis=1, equal_var=False).pvalue < 0.05

print(f"True effect: {effect_mm} mm.   Power: {significant.mean():.0%}")
print(f"Average estimate among the 'significant' trials: {estimates[significant].mean():.1f} mm "
      f"({estimates[significant].mean() / effect_mm:.1f}× the truth)")

## 🤖 Go deeper with AI
Three ways to keep learning about power and sample size: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain statistical power and the four things it depends on, using the analogy of spotting a faint band on a western blot. Then ask me to predict what happens to power when each one changes, one at a time.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Estimate the power of a two-group t-test by simulating 2,000 experiments for each n from 5 to 100 per group (effect 5 mm, SD 14 mm). Overlay the curve from statsmodels' power calculation.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *I'm planning this experiment: [describe]. Help me choose a realistic effect size and SD for a power calculation. Ask me questions first, and tell me where each number you use comes from.*

✔ **Check one claim yourself:** The simulated curve and the formula should agree within a few percentage points.

## 💬 Discuss with your partner
- Look at the shared-sheet power curve. How many patients per group would you plan for, and how many would you recruit to allow for drop-outs?
- The effect size drives everything. Where does a realistic expected effect come from: pilot data, literature, or the smallest effect that would matter clinically?
- After section E: why should a surprisingly large effect from a small study make you *more* cautious, not less?